# Inbetween Frame Generation - Training on Colab

**Prerequisites:**
1. Upload your dataset to Google Drive under `MyDrive/InBetweenGenerator/`
2. Upload this project's `model/` folder to `MyDrive/InBetweenGenerator/model/`
3. Set Runtime → Change runtime type → **T4 GPU** (or better)

**Expected Drive structure:**
```
MyDrive/InBetweenGenerator/
├── model/
│   ├── __init__.py
│   ├── dataset.py
│   ├── discriminator.py
│   ├── generator.py
│   ├── inference.py
│   ├── losses.py
│   └── train.py
└── dataset/
    └── Naruto_The_Movie_Shippuden_720p/
        ├── shot_005/
        │   ├── segment_001/
        │   │   ├── key_first.png
        │   │   ├── key_last.png
        │   │   └── inbetweens/
        │   │       ├── frame_0001.png
        │   │       └── ...
        │   └── ...
        └── ...
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Setup paths and install dependencies

In [ ]:
import sys
import os

# Path to your project on Drive
PROJECT_DIR = '/content/drive/MyDrive/InBetweenGenerator'
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset_output')  # where your shot folders are
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'training_output')  # checkpoints saved here

# Add project dir to Python path so we can import `model`
sys.path.insert(0, PROJECT_DIR)

# Verify the model package exists
assert os.path.exists(os.path.join(PROJECT_DIR, 'model', '__init__.py')), \
    f"model/ package not found in {PROJECT_DIR}. Upload model/ folder to your Drive."

print(f"Project dir: {PROJECT_DIR}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")

In [ ]:
# Colab already has torch/torchvision, but install Pillow/numpy if needed
!pip install -q Pillow>=10.0.0 numpy>=1.24.0 tqdm

## 4. (Optional) Copy dataset to local SSD for faster training

Reading directly from Drive is slow. Copy to Colab's local disk for ~5-10x faster I/O.

In [ ]:
# Set USE_LOCAL_COPY = True to copy dataset to local SSD (recommended)
USE_LOCAL_COPY = True

if USE_LOCAL_COPY:
    LOCAL_DATASET = '/content/dataset'
    if not os.path.exists(LOCAL_DATASET):
        print("Copying dataset to local SSD (this may take a while)...")
        !cp -r "{DATASET_DIR}" "{LOCAL_DATASET}"
        print("Done!")
    else:
        print("Local copy already exists.")
    TRAIN_DATASET = LOCAL_DATASET
else:
    TRAIN_DATASET = DATASET_DIR

print(f"Training from: {TRAIN_DATASET}")

## 5. Verify dataset

In [ ]:
from model.dataset import InbetweenDataset

dataset = InbetweenDataset(TRAIN_DATASET, image_size=256)
print(f"Total training samples: {len(dataset)}")

if len(dataset) == 0:
    print("\nERROR: No samples found! Check your dataset structure.")
    print(f"Expected: {TRAIN_DATASET}/<video_name>/shot_NNN/segment_NNN/")
    # List what's actually there
    for item in os.listdir(TRAIN_DATASET)[:5]:
        print(f"  Found: {item}")
else:
    # Test loading a sample
    sample = dataset[0]
    print(f"Sample keys: {list(sample.keys())}")
    print(f"Image shape: {sample['key_first'].shape}")

## 6. Train!

In [ ]:
# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 8          # Reduce to 4 if you get OOM errors
IMAGE_SIZE = 256        # Use 128 for faster experiments
LR_G = 2e-4
LR_D = 2e-4
NUM_WORKERS = 2         # Colab has limited CPU cores
SAVE_EVERY = 10
SAMPLE_EVERY = 5
RESUME = None           # Set to checkpoint path to resume, e.g.:
# RESUME = os.path.join(OUTPUT_DIR, 'checkpoints', 'latest.pt')

In [ ]:
import argparse
from model.train import train

# Build args namespace (simulates command-line arguments)
args = argparse.Namespace(
    dataset=TRAIN_DATASET,
    output=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    lr_g=LR_G,
    lr_d=LR_D,
    lambda_l1=10.0,
    lambda_perc=1.0,
    lambda_adv=1.0,
    num_workers=NUM_WORKERS,
    save_every=SAVE_EVERY,
    sample_every=SAMPLE_EVERY,
    resume=RESUME,
)

# Start training
train(args)

## 7. View training samples

In [ ]:
from IPython.display import Image, display
from pathlib import Path

samples_dir = Path(OUTPUT_DIR) / 'samples'
if samples_dir.exists():
    sample_files = sorted(samples_dir.glob('*.png'))
    if sample_files:
        print(f"Showing latest sample: {sample_files[-1].name}")
        display(Image(filename=str(sample_files[-1]), width=800))
    else:
        print("No samples generated yet.")
else:
    print("Samples directory not found. Training hasn't generated samples yet.")

## 8. Resume training after disconnect

If Colab disconnects, just re-run cells 1-4, then:

In [ ]:
# Uncomment to resume from last checkpoint
# RESUME = os.path.join(OUTPUT_DIR, 'checkpoints', 'latest.pt')
# args.resume = RESUME
# train(args)